# NLP Service — Endpoint Tests

Tests each endpoint in the NLP pipeline in isolation, then runs a full end-to-end
pass over a single article to validate the complete orchestration flow.

**Pipeline order**:
1. `POST /extract` — TF-IDF extract + MiniLM embedding
2. `POST /dedup-check` — MinHash text fingerprint
3. `POST /dedup-check-embed` — FAISS vector lookup (pre-computed embedding)
4. `POST /summarize` — LLM headline + summary + summary embedding
5. `POST /geotag` — NER + city/street/point resolution
6. `POST /classify` — Topic NLI + scope NLI (joint fusion)

**Prerequisites**: NLP service running at `NLP_SERVICE_URL` (default: `http://localhost:8000`).
Start with: `docker compose up nlp`

In [1]:
import json, sys, pprint
import httpx

sys.path.insert(0, '.')
from _scorecard import NLP_BASE_URL, HEADERS

client = httpx.Client(base_url=NLP_BASE_URL, headers=HEADERS, timeout=180.0)

def call(method: str, path: str, payload: dict) -> dict:
    """POST/GET with error display."""
    r = client.request(method, path, json=payload)
    if r.status_code >= 400:
        print(f"ERROR {r.status_code}: {r.text[:500]}")
        r.raise_for_status()
    return r.json()

# ── Sample article ─────────────────────────────────────────────────────────────
ARTICLE = {
    "article_id": "test-001",
    "headline": "Madrid ampliará su red de carriles bici en 50 kilómetros",
    "raw_text": (
        "El Ayuntamiento de Madrid ha anunciado hoy la ampliación de la red ciclista "
        "en 50 kilómetros adicionales durante 2025. La inversión prevista asciende a "
        "12 millones de euros, financiados parcialmente por el Ministerio de Transportes. "
        "Las nuevas infraestructuras conectarán los distritos de Vallecas y Carabanchel "
        "con el centro de la ciudad. El alcalde destacó que el objetivo es doblar el "
        "número de ciclistas diarios antes de 2026. La Federación de Ciclismo de Madrid "
        "celebró la medida pero reclamó más vigilancia policial en el Paseo del Prado. "
        "Según el plan, las obras en la Calle de Alcalá comenzarán en marzo."
    ),
    "source": "El País",
    "search_tags": ["carril bici", "infraestructura ciclista"],
}

try:
    health = client.get("/health").json()
    print(f"Service reachable: {health}")
except Exception as e:
    print(f"WARNING: Service not reachable ({e})\nStart the NLP service before running these cells.")

Start the NLP service before running these cells.


## 1. POST /extract

Runs TF-IDF sentence ranking on `raw_text` to produce a 200-word extract,
then encodes it with MiniLM-L12 (384-dim).

**Returns**: `extract` (str), `embedding_raw` (list[float], 384 dims)

In [2]:
extract_resp = call("POST", "/extract", {
    "article_id": ARTICLE["article_id"],
    "text": ARTICLE["raw_text"],
})

extract_text    = extract_resp["extract"]
embedding_raw   = extract_resp["embedding_raw"]

print(f"Extract ({len(extract_text.split())} words):")
print(f"  {extract_text[:300]}...")
print(f"\nEmbedding: {len(embedding_raw)}-dim  "
      f"min={min(embedding_raw):.4f}  max={max(embedding_raw):.4f}")

ERROR 401: <html>
<head><title>401 Authorization Required</title></head>
<body>
<center><h1>401 Authorization Required</h1></center>
<hr><center>nginx</center>
</body>
</html>



HTTPStatusError: Client error '401 Unauthorized' for url 'https://wiig.dia.fi.upm.es/b4c_nlp/extract'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

## 2. POST /dedup-check

MinHash LSH text fingerprint.  First call should return `duplicate_of: null`.
A second call with the same text should return the original `article_id`.

**Returns**: `duplicate_of` (str | null)

In [3]:
dedup_resp = call("POST", "/dedup-check", {
    "article_id": ARTICLE["article_id"],
    "text": ARTICLE["raw_text"],
})
print(f"First call → duplicate_of: {dedup_resp['duplicate_of']}  (expected: null)")

ERROR 401: <html>
<head><title>401 Authorization Required</title></head>
<body>
<center><h1>401 Authorization Required</h1></center>
<hr><center>nginx</center>
</body>
</html>



HTTPStatusError: Client error '401 Unauthorized' for url 'https://wiig.dia.fi.upm.es/b4c_nlp/dedup-check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

In [4]:
# Same text, different article_id → should detect duplicate
dedup_resp2 = call("POST", "/dedup-check", {
    "article_id": "test-001-copy",
    "text": ARTICLE["raw_text"],
})
print(f"Second call (copy) → duplicate_of: {dedup_resp2['duplicate_of']}  "
      f"(expected: test-001)")

ERROR 401: <html>
<head><title>401 Authorization Required</title></head>
<body>
<center><h1>401 Authorization Required</h1></center>
<hr><center>nginx</center>
</body>
</html>



HTTPStatusError: Client error '401 Unauthorized' for url 'https://wiig.dia.fi.upm.es/b4c_nlp/dedup-check'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

## 3. POST /dedup-check-embed

FAISS vector lookup using the **pre-computed** `embedding_raw` from `/extract`.
The orchestrator sends the embedding it already has — no re-encoding inside this endpoint.

**Returns**: `duplicate_of` (str | null)

In [ ]:
dedup_embed_resp = call("POST", "/dedup-check-embed", {
    "article_id": ARTICLE["article_id"],
    "embedding_raw": embedding_raw,   # ← from /extract, not re-encoded here
})
print(f"Semantic dedup → duplicate_of: {dedup_embed_resp['duplicate_of']}  (expected: null for new article)")

In [ ]:
# Near-duplicate: same content, different phrasing — cosine threshold is 0.92
near_dup_text = (
    "Madrid va a ampliar sus carriles bici en 50 km durante 2025. "
    "La inversión es de 12 millones con apoyo del Ministerio de Transportes. "
    "Los nuevos tramos unirán Vallecas y Carabanchel con el centro. "
    "El alcalde quiere duplicar los ciclistas antes de 2026."
)

near_dup_extract = call("POST", "/extract", {
    "article_id": "test-001-neardup",
    "text": near_dup_text,
})

near_dup_resp = call("POST", "/dedup-check-embed", {
    "article_id": "test-001-neardup",
    "embedding_raw": near_dup_extract["embedding_raw"],
})
print(f"Near-duplicate → duplicate_of: {near_dup_resp['duplicate_of']}")
print("(May or may not trigger depending on paraphrase similarity — cosine threshold 0.92)")

## 4. POST /summarize

Sends `text` + `extract` (from `/extract`) to Ollama for a structured Spanish
summary.  Returns an improved headline, a 2-3 sentence summary, and a 384-dim
summary embedding (used later for semantic search, not dedup).

**Returns**: `headline` (str), `summary` (str), `embedding_summary` (list[float], 384 dims)

> Requires Ollama running as a sidecar. If not available, this cell will fail with a 503.

In [ ]:
summarize_resp = call("POST", "/summarize", {
    "article_id": ARTICLE["article_id"],
    "text": ARTICLE["raw_text"],
    "extract": extract_text,            # ← from /extract
    "headline": ARTICLE["headline"],    # ← original headline as fallback seed
})

new_headline      = summarize_resp["headline"]
summary           = summarize_resp["summary"]
embedding_summary = summarize_resp["embedding_summary"]

print(f"Headline:  {new_headline}")
print(f"\nSummary ({len(summary.split())} words):\n  {summary}")
print(f"\nSummary embedding: {len(embedding_summary)}-dim")

## 5. POST /geotag

Runs the RoBERTa NER model on `text` + `headline`, then resolves each span
through the 5-level gazetteer cascade:
1. Direct GeoNames match above confidence threshold
2. Dominant document city
3. Sentence co-occurrence with resolved span
4. Source city prior
5. GeoNames lat/lon fallback → `geo_points` (kept for map plotting)

**Returns**: `geo_cities` (list), `geo_streets` (list), `geo_points` (list)

In [ ]:
geotag_resp = call("POST", "/geotag", {
    "article_id": ARTICLE["article_id"],
    "text": ARTICLE["raw_text"],
    "headline": new_headline,   # ← LLM-improved headline from /summarize
})

geo_cities  = geotag_resp["geo_cities"]
geo_streets = geotag_resp["geo_streets"]
geo_points  = geotag_resp["geo_points"]

print(f"geo_cities ({len(geo_cities)}):")
for c in geo_cities:
    print(f"  city_id={c['city_id']}  name={c['city_name']}  conf={c['confidence']:.3f}")

print(f"\ngeo_streets ({len(geo_streets)}):")
for s in geo_streets:
    print(f"  span='{s['span']}'  city_id={s['city_id']}  edges={s['edge_ids'][:3]}...")

print(f"\ngeo_points ({len(geo_points)})  [unresolved spans kept for map plotting]:")
for p in geo_points:
    print(f"  span='{p['span']}'  lat={p['lat']}  lon={p['lon']}  geonames_id={p['geonames_id']}")

## 6. POST /classify

Two separate NLI passes (critical — scope and topic must not share a softmax):
- **Topic pass**: premise = `search_tags prefix + summary`; multi-label, independent thresholds
- **Scope pass**: premise = `summary + city_context + source_context`; 3-way exclusive (national / regional / city)

The scope NLI receives all evidence simultaneously (`summary` + resolved `geo_cities` +
`source_profile`) — this is the joint fusion that runs after `/geotag`.

**Returns**: `topics` (list[str]), `scores` (dict[str, float]), `geo_scope` (str)

In [ ]:
classify_resp = call("POST", "/classify", {
    "article_id": ARTICLE["article_id"],
    "summary": summary,              # ← from /summarize
    "geo_cities": geo_cities,        # ← from /geotag (joint scope fusion)
    "search_tags": ARTICLE["search_tags"],   # ← scraper tags → topic premise
    "source_profile": {
        "city": None,
        "region": None,
        "topics": ["movilidad urbana", "infraestructura ciclista"],
    },
})

print(f"geo_scope:  {classify_resp['geo_scope']}")
print(f"\ntopics ({len(classify_resp['topics'])}):\n  {classify_resp['topics']}")
print("\nAll scores (sorted):")
for label, score in sorted(classify_resp["scores"].items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 30)
    print(f"  {label:<35} {score:.3f}  {bar}")

## 7. Scope fusion — multi-city vs single-city

Verifies that the scope NLI correctly uses `geo_cities` evidence.
A national article with multiple cities should score differently from
a city-scoped article with a single city.

In [ ]:
national_summary = (
    "El Ministerio de Transportes lanza un plan nacional de 200 millones para "
    "ampliar carriles bici en Madrid, Barcelona, Sevilla, Valencia y Bilbao."
)

city_summary = (
    "El Ayuntamiento de Sevilla inaugura el nuevo carril bici del Paseo de Colón, "
    "conectando el centro con Triana."
)

multi_cities = [
    {"city_id": 1, "city_name": "Madrid", "confidence": 0.91},
    {"city_id": 2, "city_name": "Barcelona", "confidence": 0.88},
    {"city_id": 3, "city_name": "Sevilla", "confidence": 0.85},
    {"city_id": 4, "city_name": "Valencia", "confidence": 0.84},
    {"city_id": 5, "city_name": "Bilbao", "confidence": 0.82},
]

single_city = [
    {"city_id": 3, "city_name": "Sevilla", "confidence": 0.92},
]

for label, summ, cities in [
    ("national (5 cities)", national_summary, multi_cities),
    ("city (1 city)",       city_summary,     single_city),
]:
    r = call("POST", "/classify", {
        "article_id": f"scope-test-{label[:3]}",
        "summary": summ,
        "geo_cities": cities,
        "search_tags": [],
        "source_profile": None,
    })
    print(f"[{label}] → geo_scope={r['geo_scope']}  topics={r['topics']}")

## 8. Full pipeline — end-to-end

Runs a fresh article through all 6 steps, accumulating results in memory
the same way the orchestrator (`070_ingest_news.py`) does.
Prints the complete DB payload that would be written at the end.

In [ ]:
import time

FRESH = {
    "article_id": "e2e-001",
    "headline": "Sevilla inaugurará 15 km de carriles bici nuevos antes del verano",
    "raw_text": (
        "La ciudad de Sevilla va a poner en servicio 15 kilómetros de nuevos "
        "carriles bici antes del mes de junio, según anunció hoy el Ayuntamiento. "
        "Las actuaciones se concentran en la avenida de la Constitución y en el "
        "Paseo de Colón. El coste total es de 4,5 millones de euros, cofinanciados "
        "con fondos europeos del programa FEDER. La nueva infraestructura conectará "
        "Triana con el barrio de Santa Cruz. El concejal de Movilidad destacó que "
        "Sevilla ya es la ciudad española con mayor proporción de ciclistas urbanos."
    ),
    "source": "Diario de Sevilla",
    "search_tags": ["carril bici", "movilidad urbana"],
}

timings = {}
result  = {"article_id": FRESH["article_id"]}

# Step 1: extract
t0 = time.perf_counter()
r = call("POST", "/extract", {"article_id": FRESH["article_id"], "text": FRESH["raw_text"]})
timings["extract"] = time.perf_counter() - t0
result["extract"]      = r["extract"]
result["embedding_raw"] = r["embedding_raw"]

# Step 2: text dedup
t0 = time.perf_counter()
r = call("POST", "/dedup-check", {"article_id": FRESH["article_id"], "text": FRESH["raw_text"]})
timings["dedup_check"] = time.perf_counter() - t0
if r["duplicate_of"]:
    print(f"TEXT DUPLICATE of {r['duplicate_of']} — stopping")
else:
    print("Step 2 dedup-check: not a duplicate")

# Step 3: embed dedup
t0 = time.perf_counter()
r = call("POST", "/dedup-check-embed", {
    "article_id": FRESH["article_id"], "embedding_raw": result["embedding_raw"]
})
timings["dedup_embed"] = time.perf_counter() - t0
if r["duplicate_of"]:
    print(f"SEMANTIC DUPLICATE of {r['duplicate_of']} — stopping")
else:
    print("Step 3 dedup-check-embed: not a duplicate")

# Step 4: summarize
t0 = time.perf_counter()
r = call("POST", "/summarize", {
    "article_id": FRESH["article_id"],
    "text": FRESH["raw_text"],
    "extract": result["extract"],
    "headline": FRESH["headline"],
})
timings["summarize"] = time.perf_counter() - t0
result["headline"]          = r["headline"]
result["summary"]           = r["summary"]
result["embedding_summary"] = r["embedding_summary"]
print(f"Step 4 summarize: '{result['headline']}'")

# Step 5: geotag
t0 = time.perf_counter()
r = call("POST", "/geotag", {
    "article_id": FRESH["article_id"],
    "text": FRESH["raw_text"],
    "headline": result["headline"],
})
timings["geotag"] = time.perf_counter() - t0
result["geo_cities"]  = r["geo_cities"]
result["geo_streets"] = r["geo_streets"]
result["geo_points"]  = r["geo_points"]
print(f"Step 5 geotag: cities={[c['city_name'] for c in result['geo_cities']]}  "
      f"streets={len(result['geo_streets'])}  points={len(result['geo_points'])}")

# Step 6: classify (joint scope fusion — receives geo_cities from step 5)
t0 = time.perf_counter()
r = call("POST", "/classify", {
    "article_id": FRESH["article_id"],
    "summary": result["summary"],
    "geo_cities": result["geo_cities"],
    "search_tags": FRESH["search_tags"],
    "source_profile": None,
})
timings["classify"] = time.perf_counter() - t0
result["topics"]    = r["topics"]
result["scores"]    = r["scores"]
result["geo_scope"] = r["geo_scope"]
print(f"Step 6 classify: scope={result['geo_scope']}  topics={result['topics']}")

In [ ]:
# ── DB payload summary ──────────────────────────────────────────────────────────
print("=" * 60)
print("DB payload (what the orchestrator would INSERT):")
print("=" * 60)
print(f"  article_id:        {result['article_id']}")
print(f"  headline:          {result.get('headline', '—')}")
print(f"  summary:           {result.get('summary', '—')[:120]}...")
print(f"  embedding_raw:     [{len(result.get('embedding_raw', []))} floats]")
print(f"  embedding_summary: [{len(result.get('embedding_summary', []))} floats]")
print(f"  geo_cities:        {result.get('geo_cities', [])}")
print(f"  geo_streets:       {result.get('geo_streets', [])}")
print(f"  geo_points:        {result.get('geo_points', [])}")
print(f"  topics:            {result.get('topics', [])}")
print(f"  geo_scope:         {result.get('geo_scope', '—')}")

print("\n── Step latencies ─────────────────────────────────────")
for step, t in timings.items():
    print(f"  {step:<20} {t:.2f}s")
print(f"  {'total':<20} {sum(timings.values()):.2f}s")

## 9. Edge cases

Verifies graceful handling of minimal and unusual inputs.

In [ ]:
# 9a — very short text (< 3 sentences)
short = call("POST", "/extract", {
    "article_id": "edge-short",
    "text": "Madrid tendrá más carriles bici.",
})
print(f"Short text → extract='{short['extract']}'  emb_dim={len(short['embedding_raw'])}")

In [ ]:
# 9b — text with no Spanish cities → geo_cities should be empty, scope fallback
no_cities = call("POST", "/geotag", {
    "article_id": "edge-nocity",
    "text": "El gobierno aprobó nuevas normativas para patinetes eléctricos en zonas peatonales.",
    "headline": "Nueva regulación para patinetes",
})
print(f"No-city text → geo_cities={no_cities['geo_cities']}  "
      f"geo_points={no_cities['geo_points']}")

In [ ]:
# 9c — classify with empty geo_cities → scope should default via NLI text signal
no_geo_classify = call("POST", "/classify", {
    "article_id": "edge-noscope",
    "summary": "España aprueba una ley nacional de movilidad urbana sostenible.",
    "geo_cities": [],
    "search_tags": [],
    "source_profile": None,
})
print(f"Empty geo_cities → geo_scope={no_geo_classify['geo_scope']}  "
      f"(expected: national)  topics={no_geo_classify['topics']}")

In [ ]:
# 9d — search_tags enrichment: same summary, with vs without tags
base_summary = "Se instalan nuevas señales de tráfico en el centro urbano."

without_tags = call("POST", "/classify", {
    "article_id": "edge-notags",
    "summary": base_summary,
    "geo_cities": [],
    "search_tags": [],
    "source_profile": None,
})

with_tags = call("POST", "/classify", {
    "article_id": "edge-withtags",
    "summary": base_summary,
    "geo_cities": [],
    "search_tags": ["seguridad vial", "señalización"],
    "source_profile": None,
})

print(f"Without tags → topics={without_tags['topics']}")
print(f"With tags    → topics={with_tags['topics']}")
print("(Tags should nudge the topic NLI toward matching labels)")